<a href="https://colab.research.google.com/github/radhikatyagi388/Ai_60_Day_Challange/blob/main/Day_27_Structured_Tool_Calling_with_the_OpenAI_Functions_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 🚀 Day 27 — Structured Tool Calling Agent

> **Focus Area:** Structured Tool Calling
> **Technologies:** Python, FastAPI, Pydantic
> **Environment:** Google Colab
> **API:** No OpenAI API required

---

## 🎯 Objective

In **Day 26**, I built a manual ReAct agent that identified tool calls by parsing the **raw text generated by the LLM**.

For example:

```text
TOOL: calculate
ARGS: {"expression": "20+30"}

In [2]:
# ============================================================
# DAY 27 — STRUCTURED TOOL CALLING
# WITHOUT OPENAI API
# Single Google Colab Cell
# ============================================================

!pip -q install fastapi uvicorn nest-asyncio pydantic requests


import json
import ast
import operator
import datetime
import threading
import time
import requests
import nest_asyncio

from typing import Any, Dict
from pydantic import BaseModel, Field, ValidationError
from fastapi import FastAPI


# ============================================================
# 1. FOUR TOOLS
# ============================================================

DOCUMENTS = {
    "refund policy":
        "Customers can request a refund within 30 days of purchase.",

    "password reset":
        "Go to Settings > Security > Reset Password.",

    "shipping policy":
        "Standard shipping takes 5-7 business days. "
        "Express shipping takes 1-2 business days.",

    "student discount":
        "Students receive a 15 percent discount after verification.",

    "leave policy":
        "Employees receive 18 paid leave days per year."
}


def search_documents(query: str) -> str:

    query = query.lower()

    results = []

    for title, content in DOCUMENTS.items():

        if any(
            word in title.lower() or word in content.lower()
            for word in query.split()
        ):
            results.append({
                "title": title,
                "content": content
            })

    if not results:

        return json.dumps({
            "found": False,
            "message": "No document found."
        })

    return json.dumps({
        "found": True,
        "results": results[:3]
    })


def get_weather_stub(city: str) -> str:

    weather = {
        "delhi": {
            "temperature": 31,
            "condition": "Sunny"
        },

        "meerut": {
            "temperature": 30,
            "condition": "Partly Cloudy"
        },

        "london": {
            "temperature": 16,
            "condition": "Cloudy"
        }
    }

    data = weather.get(
        city.lower(),
        {
            "temperature": 25,
            "condition": "Unknown"
        }
    )

    return json.dumps({
        "city": city,
        **data
    })


def calculate(expression: str) -> str:

    allowed = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Mod: operator.mod,
        ast.Pow: operator.pow,
        ast.USub: operator.neg
    }

    def evaluate(node):

        if isinstance(node, ast.Expression):
            return evaluate(node.body)

        if isinstance(node, ast.Constant):

            if isinstance(node.value, (int, float)):
                return node.value

            raise ValueError("Only numbers allowed.")

        if isinstance(node, ast.BinOp):

            left = evaluate(node.left)
            right = evaluate(node.right)

            op = allowed.get(type(node.op))

            if not op:
                raise ValueError("Operator not allowed.")

            return op(left, right)

        if isinstance(node, ast.UnaryOp):

            value = evaluate(node.operand)

            op = allowed.get(type(node.op))

            if not op:
                raise ValueError("Operator not allowed.")

            return op(value)

        raise ValueError("Invalid expression.")

    try:

        tree = ast.parse(expression, mode="eval")

        result = evaluate(tree)

        return json.dumps({
            "expression": expression,
            "result": result
        })

    except Exception as e:

        return json.dumps({
            "error": str(e)
        })


def get_today() -> str:

    return json.dumps({
        "today": datetime.date.today().isoformat()
    })


# ============================================================
# 2. FUNCTION REGISTRY
# ============================================================

FUNCTIONS = {

    "search_documents":
        search_documents,

    "get_weather_stub":
        get_weather_stub,

    "calculate":
        calculate,

    "get_today":
        get_today
}


# ============================================================
# 3. JSON TOOL SCHEMAS
# ============================================================

TOOL_SCHEMAS = [

    {
        "name": "search_documents",

        "description":
            "Search company documents and policies.",

        "parameters": {

            "type": "object",

            "properties": {

                "query": {
                    "type": "string",
                    "description":
                        "What information should be searched?"
                }
            },

            "required": ["query"]
        }
    },


    {
        "name": "get_weather_stub",

        "description":
            "Get simulated weather for a city.",

        "parameters": {

            "type": "object",

            "properties": {

                "city": {
                    "type": "string",
                    "description":
                        "Name of the city."
                }
            },

            "required": ["city"]
        }
    },


    {
        "name": "calculate",

        "description":
            "Calculate an arithmetic expression.",

        "parameters": {

            "type": "object",

            "properties": {

                "expression": {
                    "type": "string",
                    "description":
                        "Mathematical expression."
                }
            },

            "required": ["expression"]
        }
    },


    {
        "name": "get_today",

        "description":
            "Get today's date.",

        "parameters": {

            "type": "object",

            "properties": {},

            "required": []
        }
    }
]


print("=" * 70)
print("FOUR STRUCTURED TOOL SCHEMAS")
print("=" * 70)

for tool in TOOL_SCHEMAS:

    print(
        "\nTool:",
        tool["name"]
    )

    print(
        "Description:",
        tool["description"]
    )

    print(
        "Parameters:",
        json.dumps(
            tool["parameters"],
            indent=2
        )
    )


# ============================================================
# 4. ARGUMENT VALIDATION
# ============================================================

class SearchArgs(BaseModel):

    query: str = Field(
        min_length=1
    )


class WeatherArgs(BaseModel):

    city: str = Field(
        min_length=1
    )


class CalculateArgs(BaseModel):

    expression: str = Field(
        min_length=1
    )


class TodayArgs(BaseModel):
    pass


VALIDATORS = {

    "search_documents":
        SearchArgs,

    "get_weather_stub":
        WeatherArgs,

    "calculate":
        CalculateArgs,

    "get_today":
        TodayArgs
}


def validate_arguments(
    tool_name,
    arguments
):

    if tool_name not in VALIDATORS:

        raise ValueError(
            f"Unknown tool: {tool_name}"
        )

    validator = VALIDATORS[tool_name]

    try:

        obj = validator(**arguments)

        return obj.model_dump()

    except ValidationError as e:

        raise ValueError(
            f"INVALID ARGUMENTS: {e}"
        )


# ============================================================
# 5. SAFE TOOL EXECUTOR
# ============================================================

def execute_tool(
    tool_name,
    arguments
):

    validated = validate_arguments(
        tool_name,
        arguments
    )

    return FUNCTIONS[tool_name](
        **validated
    )


# ============================================================
# 6. MOCK STRUCTURED LLM
# ============================================================
#
# Instead of OpenAI API, this function behaves like an LLM
# that returns structured tool calls.
#
# IMPORTANT:
# This demonstrates the architecture without an API key.
# ============================================================


def structured_llm(question, previous_results):

    q = question.lower()


    # --------------------------------------------------------
    # TOOL 1 — SEARCH DOCUMENTS
    # --------------------------------------------------------

    if (
        ("refund" in q)
        and not any(
            x["tool"] == "search_documents"
            for x in previous_results
        )
    ):

        return {
            "type": "tool_call",

            "tool_calls": [

                {
                    "name":
                        "search_documents",

                    "arguments": {
                        "query":
                            "refund policy"
                    }
                }
            ]
        }


    # --------------------------------------------------------
    # TOOL 2 — STUDENT DISCOUNT
    # --------------------------------------------------------

    if (
        ("student" in q or "discount" in q)
        and not any(
            x["tool"] == "search_documents"
            for x in previous_results
        )
    ):

        return {
            "type": "tool_call",

            "tool_calls": [

                {
                    "name":
                        "search_documents",

                    "arguments": {
                        "query":
                            "student discount"
                    }
                }
            ]
        }


    # --------------------------------------------------------
    # WEATHER
    # --------------------------------------------------------

    if (
        "weather" in q
        and not any(
            x["tool"] == "get_weather_stub"
            for x in previous_results
        )
    ):

        city = "Delhi"

        if "meerut" in q:
            city = "Meerut"

        if "london" in q:
            city = "London"

        return {

            "type": "tool_call",

            "tool_calls": [

                {
                    "name":
                        "get_weather_stub",

                    "arguments": {
                        "city": city
                    }
                }
            ]
        }


    # --------------------------------------------------------
    # TODAY
    # --------------------------------------------------------

    if (
        "today" in q
        and not any(
            x["tool"] == "get_today"
            for x in previous_results
        )
    ):

        return {

            "type": "tool_call",

            "tool_calls": [

                {
                    "name":
                        "get_today",

                    "arguments": {}
                }
            ]
        }


    # --------------------------------------------------------
    # CALCULATOR
    # --------------------------------------------------------

    if (
        "calculate" in q
        and not any(
            x["tool"] == "calculate"
            for x in previous_results
        )
    ):

        if "20" in q and "30" in q:

            expression = "20 + 30"

        elif "2000" in q:

            expression = "2000 * 15 / 100"

        elif "100" in q and "4" in q:

            expression = "100 / 4"

        else:

            expression = "10 + 20"

        return {

            "type": "tool_call",

            "tool_calls": [

                {
                    "name":
                        "calculate",

                    "arguments": {
                        "expression":
                            expression
                    }
                }
            ]
        }


    # --------------------------------------------------------
    # SEQUENTIAL DISCOUNT CALCULATION
    # --------------------------------------------------------

    if (
        "discount amount" in q
        and any(
            x["tool"] == "search_documents"
            for x in previous_results
        )
        and not any(
            x["tool"] == "calculate"
            for x in previous_results
        )
    ):

        return {

            "type": "tool_call",

            "tool_calls": [

                {
                    "name":
                        "calculate",

                    "arguments": {
                        "expression":
                            "2000 * 15 / 100"
                    }
                }
            ]
        }


    # --------------------------------------------------------
    # FINAL RESPONSE
    # --------------------------------------------------------

    if previous_results:

        last = previous_results[-1]

        return {

            "type": "final",

            "content":
                f"Based on the tool results: "
                f"{last['result']}"
        }


    return {

        "type": "final",

        "content":
            "I can answer using the available tools."
    }


# ============================================================
# 7. STRUCTURED TOOL CALLING LOOP
# ============================================================

def run_agent(question):

    history = []

    print("\n" + "=" * 70)
    print("USER QUESTION")
    print("=" * 70)

    print(question)


    for step in range(10):

        response = structured_llm(
            question,
            history
        )


        # ----------------------------------------------------
        # FINAL RESPONSE
        # ----------------------------------------------------

        if response["type"] == "final":

            print("\n" + "=" * 70)
            print("FINAL ANSWER")
            print("=" * 70)

            print(
                response["content"]
            )

            return {

                "answer":
                    response["content"],

                "tool_calls":
                    history
            }


        # ----------------------------------------------------
        # STRUCTURED TOOL CALL
        # ----------------------------------------------------

        for call in response["tool_calls"]:

            name = call["name"]

            arguments = call["arguments"]


            print("\n" + "-" * 70)
            print("STRUCTURED TOOL CALL")
            print("-" * 70)

            print("Tool:", name)

            print(
                "Arguments:",
                json.dumps(
                    arguments,
                    indent=2
                )
            )


            try:

                result = execute_tool(
                    name,
                    arguments
                )

                status = "success"

            except Exception as e:

                result = json.dumps({
                    "error": str(e)
                })

                status = "validation_error"


            print("Result:", result)


            history.append({

                "tool":
                    name,

                "arguments":
                    arguments,

                "result":
                    result,

                "status":
                    status
            })


    return {

        "answer":
            "Maximum iterations reached.",

        "tool_calls":
            history
    }


# ============================================================
# 8. FIVE BENCHMARK PROBLEMS
# ============================================================

QUESTIONS = [

    "What is the refund policy?",

    "What is the weather in Delhi?",

    "What is today's date?",

    "Calculate 20 + 30.",

    "Find the student discount and calculate the discount amount on 2000."
]


print("\n" + "=" * 70)
print("FIVE-PROBLEM BENCHMARK")
print("=" * 70)


benchmark_results = []


for i, question in enumerate(
    QUESTIONS,
    1
):

    print(
        f"\n\nPROBLEM {i}"
    )

    result = run_agent(
        question
    )

    benchmark_results.append(
        result
    )


# ============================================================
# 9. TWO SEQUENTIAL TOOL CALL DEMO
# ============================================================

print("\n\n" + "=" * 70)
print("TWO SEQUENTIAL TOOL CALL DEMONSTRATION")
print("=" * 70)


question = (
    "Find the student discount and "
    "calculate the discount amount on 2000."
)


result = run_agent(
    question
)


print("\nFULL EXCHANGE")
print("=" * 70)


for i, call in enumerate(
    result["tool_calls"],
    1
):

    print(
        f"\nSTEP {i}"
    )

    print(
        "Tool:",
        call["tool"]
    )

    print(
        "Arguments:",
        call["arguments"]
    )

    print(
        "Result:",
        call["result"]
    )


# ============================================================
# 10. THREE INVALID ARGUMENT TESTS
# ============================================================

print("\n\n" + "=" * 70)
print("THREE INVALID ARGUMENT TESTS")
print("=" * 70)


invalid_tests = [

    (
        "search_documents",
        {}
    ),

    (
        "get_weather_stub",
        {
            "city": ""
        }
    ),

    (
        "calculate",
        {
            "expression": ""
        }
    )
]


validation_results = []


for i, (
    tool,
    args
) in enumerate(
    invalid_tests,
    1
):

    print(
        f"\nTEST {i}"
    )

    print(
        "Tool:",
        tool
    )

    print(
        "Arguments:",
        args
    )


    try:

        execute_tool(
            tool,
            args
        )

        print(
            "❌ ERROR WAS NOT CAUGHT"
        )

        validation_results.append(
            False
        )

    except Exception as e:

        print(
            "✅ VALIDATION CAUGHT:"
        )

        print(e)

        validation_results.append(
            True
        )


# ============================================================
# 11. DAY 26 MANUAL TEXT PARSING
# ============================================================

def manual_parser(text):

    """
    Day 26 style parser.

    Expected:

    TOOL: calculate
    ARGS: {"expression":"20+30"}
    """

    lines = text.splitlines()

    tool = None
    args = None


    for line in lines:

        if line.startswith(
            "TOOL:"
        ):

            tool = line[
                len("TOOL:"):
            ].strip()


        elif line.startswith(
            "ARGS:"
        ):

            args = json.loads(
                line[
                    len("ARGS:"):
                ].strip()
            )


    if not tool:

        raise ValueError(
            "Tool name not found."
        )


    if args is None:

        raise ValueError(
            "Arguments not found."
        )


    return tool, args


manual_tests = [

    'TOOL: calculate\nARGS: {"expression":"20+30"}',

    'tool: calculate\nargs: {"expression":"20+30"}',

    'CALL calculate WITH {"expression":"20+30"}',

    'TOOL: calculate',

    'I will calculate.\nTOOL: calculate\nARGS: {"expression":"20+30"}'
]


manual_success = 0


print("\n\n" + "=" * 70)
print("DAY 26 MANUAL PARSING TEST")
print("=" * 70)


for text in manual_tests:

    print("\nInput:")
    print(text)

    try:

        manual_parser(text)

        print("✅ Parsed")

        manual_success += 1

    except Exception as e:

        print("❌ Failed:")
        print(e)


manual_total = len(
    manual_tests
)


# ============================================================
# 12. RELIABILITY COMPARISON
# ============================================================

structured_success = sum(
    x["status"] == "success"
    for result in benchmark_results
    for x in result["tool_calls"]
)


structured_total = sum(
    len(result["tool_calls"])
    for result in benchmark_results
)


validation_success = sum(
    validation_results
)


comparison = f"""

# DAY 27 — STRUCTURED TOOL CALLING

## Focus Area

Structured tool calling removes the need to parse
fragile raw LLM text.

---

# Day 26 — Manual Text Parsing

Day 26 expected output like:

TOOL: calculate
ARGS: {{"expression":"20+30"}}

Python manually parsed this text.

### Problems

- Exact formatting matters.
- Tool names can be formatted differently.
- Extra natural language can break the parser.
- Missing arguments can cause errors.
- Multiple sequential calls require more parsing logic.

Example:

TOOL: calculate
ARGS: {{"expression":"20+30"}}

works.

But:

tool: calculate
args: {{"expression":"20+30"}}

may fail.

---

# Day 27 — Structured Tool Calling

The agent now works with structured data:

{{
    "type": "tool_call",
    "tool_calls": [
        {{
            "name": "calculate",
            "arguments": {{
                "expression": "20+30"
            }}
        }}
    ]
}}

The application directly accesses:

- tool name
- arguments
- tool result

There is no need to parse arbitrary natural-language output.

---

# Tool Execution Pipeline

User
 ↓
Structured LLM
 ↓
Tool Call JSON
 ↓
Argument Validation
 ↓
Python Function
 ↓
Tool Result
 ↓
Structured LLM
 ↓
Final Answer

---

# Argument Validation

Three invalid cases were tested.

1. Missing query
2. Empty city
3. Empty mathematical expression

The validation wrapper catches these before
the underlying Python functions execute.

Validation caught:
{validation_success}/3

---

# Sequential Tool Calls

The agent demonstrated:

search_documents
        ↓
student discount = 15%
        ↓
calculate
        ↓
2000 × 15 / 100
        ↓
300

This demonstrates multi-step tool execution.

---

# Reliability Comparison

## Day 26

Manual text parsing:

LLM text
 ↓
Custom parser
 ↓
Tool name
 ↓
Argument parser
 ↓
Function

More formatting-dependent.

## Day 27

Structured calling:

Structured response
 ↓
Tool name
 ↓
JSON arguments
 ↓
Validation
 ↓
Function

Much less dependent on exact textual formatting.

---

# Important Note

This Colab implementation uses a MOCK structured LLM
instead of a real OpenAI API.

Therefore it demonstrates the architecture and execution
mechanism without requiring an API key.

In a production system, the `structured_llm()` function
would be replaced by an actual LLM/API that supports
structured function calling.

---

# FastAPI

Endpoint:

POST /agent

Input:

{{"question": "What is the refund policy?"}}

Output:

{{
    "answer": "...",
    "tool_calls": [...]
}}

---

# Conclusion

Structured tool calling is more reliable than manual
text parsing because the application receives explicit
tool names and structured arguments rather than depending
on a manually defined text format.

Day 27 therefore improves the reliability and scalability
of the Day 26 ReAct agent architecture.

"""


with open(
    "/content/day27_reliability_comparison.md",
    "w"
) as f:

    f.write(comparison)


# ============================================================
# 13. FASTAPI
# ============================================================

app = FastAPI(
    title="Day 27 Structured Tool Calling Agent",
    version="1.0"
)


class AgentRequest(BaseModel):

    question: str = Field(
        min_length=1
    )


@app.post("/agent")
def agent_endpoint(
    request: AgentRequest
):

    return run_agent(
        request.question
    )


# ============================================================
# 14. START FASTAPI
# ============================================================

nest_asyncio.apply()


def start_server():

    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000
    )


thread = threading.Thread(
    target=start_server,
    daemon=True
)

thread.start()


time.sleep(3)


# ============================================================
# 15. TEST API
# ============================================================

print("\n\n" + "=" * 70)
print("FASTAPI TEST")
print("=" * 70)


try:

    response = requests.post(

        "http://127.0.0.1:8000/agent",

        json={
            "question":
                "What is the refund policy?"
        },

        timeout=60
    )


    print(
        "Status:",
        response.status_code
    )

    print(
        json.dumps(
            response.json(),
            indent=2
        )
    )


except Exception as e:

    print(
        "API test error:",
        e
    )


# ============================================================
# 16. FINAL SUMMARY
# ============================================================

print("\n\n" + "=" * 70)
print("🎉 DAY 27 COMPLETE")
print("=" * 70)

print("""
✅ Four JSON tool schemas

   • search_documents
   • get_weather_stub
   • calculate
   • get_today

✅ Structured tool execution loop

✅ Five benchmark problems

✅ Two sequential tool calls

✅ Argument validation wrapper

✅ Three invalid argument tests

✅ Day 26 manual parser simulation

✅ Day 26 vs Day 27 reliability comparison

✅ FastAPI POST /agent

✅ Complete tool-call history

✅ Comparison document generated

--------------------------------------------------

FastAPI:
http://127.0.0.1:8000

Swagger:
http://127.0.0.1:8000/docs

Comparison document:
/content/day27_reliability_comparison.md
""")

FOUR STRUCTURED TOOL SCHEMAS

Tool: search_documents
Description: Search company documents and policies.
Parameters: {
  "type": "object",
  "properties": {
    "query": {
      "type": "string",
      "description": "What information should be searched?"
    }
  },
  "required": [
    "query"
  ]
}

Tool: get_weather_stub
Description: Get simulated weather for a city.
Parameters: {
  "type": "object",
  "properties": {
    "city": {
      "type": "string",
      "description": "Name of the city."
    }
  },
  "required": [
    "city"
  ]
}

Tool: calculate
Description: Calculate an arithmetic expression.
Parameters: {
  "type": "object",
  "properties": {
    "expression": {
      "type": "string",
      "description": "Mathematical expression."
    }
  },
  "required": [
    "expression"
  ]
}

Tool: get_today
Description: Get today's date.
Parameters: {
  "type": "object",
  "properties": {},
  "required": []
}

FIVE-PROBLEM BENCHMARK


PROBLEM 1

USER QUESTION
What is the refund po

INFO:     Started server process [497]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)




FASTAPI TEST

USER QUESTION
What is the refund policy?

----------------------------------------------------------------------
STRUCTURED TOOL CALL
----------------------------------------------------------------------
Tool: search_documents
Arguments: {
  "query": "refund policy"
}
Result: {"found": true, "results": [{"title": "refund policy", "content": "Customers can request a refund within 30 days of purchase."}, {"title": "shipping policy", "content": "Standard shipping takes 5-7 business days. Express shipping takes 1-2 business days."}, {"title": "leave policy", "content": "Employees receive 18 paid leave days per year."}]}

FINAL ANSWER
Based on the tool results: {"found": true, "results": [{"title": "refund policy", "content": "Customers can request a refund within 30 days of purchase."}, {"title": "shipping policy", "content": "Standard shipping takes 5-7 business days. Express shipping takes 1-2 business days."}, {"title": "leave policy", "content": "Employees receive 18 p

In [ ]:
                         USER
                           │
                           ▼
                  ┌─────────────────┐
                  │ Structured LLM  │
                  └────────┬────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │  Tool Selection │
                  └────────┬────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │ JSON Arguments  │
                  └────────┬────────┘
                           │
                           ▼
                  ┌─────────────────┐
                  │   Validation    │
                  │    Pydantic     │
                  └────────┬────────┘
                           │
              ┌────────────┼────────────┐
              │            │            │
              ▼            ▼            ▼
          Search        Weather     Calculate
              │            │            │
              └────────────┼────────────┘
                           │
                           ▼
                     Tool Result
                           │
                           ▼
                  ┌─────────────────┐
                  │ Structured LLM  │
                  └────────┬────────┘
                           │
                           ▼
                     FINAL ANSWER